# Mixed-Type chẩn đoán TRAIN (fit mẫu dương của fine-tune)

**Chẩn đoán CHỈ TRÊN TRAIN** (không phải kết quả chính). Câu hỏi: mô hình B (kiến trúc và đầu vào giữ nguyên) có fit được các mẫu dương TRAIN của một fold khi cho học lâu hơn / lr cao hơn không? Nếu fit được → cấu hình fine-tune cũ (6 epoch, lr nhỏ) quá ngắn; nếu vẫn không fit được dù học lâu → đầu vào không đủ tách. Fit được TRAIN **không** chứng minh khái quát hóa.

Không đọc mẫu test của fold, không đánh giá OpTC TEST, không chọn gì theo chúng. Thiết kế ở `02_artifacts/34_train_diag_v1/STEP_34_TRAIN_DIAGNOSTIC.md`.

**Gắn đúng 2 dataset:** `v2_network_dl_kaggle_upload` (replay OpTC) và `v3_ft_kaggle_upload`. Bật GPU.

In [ ]:
import os, sys, json, shutil
from pathlib import Path
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())


## Cấu hình
Mỗi cấu hình = (lr, replay) chạy `EPOCHS` epoch trên TRAIN của fold `FOLD`, mở khóa toàn bộ tham số ngay từ đầu (khác giai đoạn A/B cũ), đo độ khớp TRAIN mỗi `EVAL_EVERY` epoch.

In [ ]:
MODEL = 'mixed'
FOLD = 0
EPOCHS = 100
EVAL_EVERY = 10
CONFIGS = [["lr1e-4_noreplay", 0.0001, 0], ["lr3e-4_noreplay", 0.0003, 0], ["lr3e-4_replay600", 0.0003, 600]]   # (tag, lr, replay mẫu/epoch)
RUN_NAME = f'{MODEL}_traindiag_fold{FOLD}'
print(MODEL, FOLD, EPOCHS, CONFIGS)


In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path


def unwrap_same_name_folder(
    root: Path,
    name: str,
) -> Path:
    path = root / name

    if not path.exists():
        raise FileNotFoundError(
            f"Missing folder: {path}"
        )

    while (path / name).is_dir():
        path = path / name

    return path


def resolve_kaggle_dataset_root(
    *,
    input_root: str | Path = "/kaggle/input",
    working_root: str | Path = "/kaggle/working/optc_network_dl",
) -> Path:
    input_root = Path(input_root)

    manifests = list(
        input_root.rglob(
            "dataset_manifest.json"
        )
    )

    if not manifests:
        raise FileNotFoundError(
            "Cannot find dataset_manifest.json under /kaggle/input"
        )

    if len(manifests) > 1:
        raise RuntimeError(
            "Nhieu dataset_manifest.json duoc tim thay (vd. dataset LOSO co 3 holdout). "
            "Chi gan MOT dataset vao notebook:\n"
            + "\n".join(str(m) for m in manifests)
        )

    source_root = None

    for manifest in manifests:
        candidate = manifest.parent

        if (
            candidate
            / "build_config.json"
        ).exists():
            source_root = candidate
            break

    if source_root is None:
        raise RuntimeError(
            "Found dataset_manifest.json but not build_config.json"
        )

    working_root = Path(working_root)

    if working_root.exists():
        shutil.rmtree(working_root)

    working_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filename in (
        "dataset_manifest.json",
        "build_config.json",
        "README_DATASET.md",
    ):
        src = source_root / filename

        if src.exists():
            (
                working_root / filename
            ).symlink_to(src)

    for folder in (
        "shards",
        "split",
        "preprocessing",
        "integrity",
    ):
        real_folder = unwrap_same_name_folder(
            source_root,
            folder,
        )

        (
            working_root / folder
        ).symlink_to(
            real_folder,
            target_is_directory=True,
        )

    required = [
        "dataset_manifest.json",
        "build_config.json",
        "split/network_sample_manifest.csv",
        "preprocessing/network_category_maps.json",
        "preprocessing/network_numeric_stats.json",
        "preprocessing/network_preprocess_spec.json",
        "preprocessing/network_preprocess_fingerprint.json",
        "shards/train",
        "shards/val",
        "shards/test",
    ]

    for relative in required:
        path = working_root / relative

        print(
            f"{'PASS' if path.exists() else 'FAIL':4s}",
            relative,
        )

        if not path.exists():
            raise FileNotFoundError(path)

    return working_root


DATASET_ROOT = resolve_kaggle_dataset_root()
print('\nDATASET_ROOT =', DATASET_ROOT)
manifest = json.loads((DATASET_ROOT/'dataset_manifest.json').read_text())
print(json.dumps(manifest['split_stats'], indent=2))


In [ ]:
ft_manifests = list(Path('/kaggle/input').rglob('ft_manifest.json'))
assert len(ft_manifests) == 1, ft_manifests
FT_ROOT = ft_manifests[0].parent
print('FT_ROOT =', FT_ROOT)


In [ ]:
WORK_CODE = Path('/kaggle/working/diag_code')
WORK_CODE.mkdir(parents=True, exist_ok=True)
(WORK_CODE/'mixed_type_attention_model.py').write_text('from __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport torch\nimport torch.nn as nn\n\n\n@dataclass(frozen=True)\nclass MixedTypeModelConfig:\n    categorical_cardinalities: list[int]\n    numerical_dim: int\n    binary_dim: int = 1\n    max_events: int = 512\n    d_model: int = 64\n    num_heads: int = 4\n    feature_layers: int = 1\n    temporal_layers: int = 2\n    ff_dim: int = 128\n    dropout: float = 0.2\n\n\nclass SelfAttentionBlock(nn.Module):\n    def __init__(\n        self,\n        *,\n        d_model: int,\n        num_heads: int,\n        ff_dim: int,\n        dropout: float,\n    ):\n        super().__init__()\n\n        if d_model % num_heads != 0:\n            raise ValueError(\n                "d_model must be divisible by num_heads"\n            )\n\n        self.norm1 = nn.LayerNorm(d_model)\n        self.attn = nn.MultiheadAttention(\n            d_model,\n            num_heads,\n            dropout=dropout,\n            batch_first=True,\n        )\n        self.attn_dropout = nn.Dropout(dropout)\n\n        self.norm2 = nn.LayerNorm(d_model)\n        self.ff = nn.Sequential(\n            nn.Linear(d_model, ff_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(ff_dim, d_model),\n            nn.Dropout(dropout),\n        )\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        *,\n        key_padding_mask: torch.Tensor | None = None,\n        return_attention: bool = False,\n    ):\n        y = self.norm1(x)\n\n        attended, weights = self.attn(\n            y,\n            y,\n            y,\n            key_padding_mask=key_padding_mask,\n            need_weights=return_attention,\n            average_attn_weights=False,\n        )\n\n        x = x + self.attn_dropout(attended)\n        x = x + self.ff(self.norm2(x))\n\n        return x, weights\n\n\nclass MixedTypeEventEncoder(nn.Module):\n    TYPE_CAT = 0\n    TYPE_NUM = 1\n    TYPE_BIN = 2\n    TYPE_DELTA = 3\n\n    def __init__(\n        self,\n        *,\n        categorical_cardinalities: list[int],\n        numerical_dim: int,\n        binary_dim: int,\n        d_model: int,\n        num_heads: int,\n        num_layers: int,\n        ff_dim: int,\n        dropout: float,\n    ):\n        super().__init__()\n\n        self.categorical_dim = len(categorical_cardinalities)\n        self.numerical_dim = int(numerical_dim)\n        self.binary_dim = int(binary_dim)\n\n        if self.binary_dim != 1:\n            raise ValueError(\n                "Current baseline expects one binary field."\n            )\n\n        self.cat_embeddings = nn.ModuleList([\n            nn.Embedding(int(cardinality), d_model)\n            for cardinality in categorical_cardinalities\n        ])\n\n        self.num_weight = nn.Parameter(\n            torch.empty(self.numerical_dim, d_model)\n        )\n        self.num_bias = nn.Parameter(\n            torch.zeros(self.numerical_dim, d_model)\n        )\n        self.num_missing_vector = nn.Parameter(\n            torch.zeros(self.numerical_dim, d_model)\n        )\n\n        self.binary_embedding = nn.Embedding(2, d_model)\n\n        self.delta_weight = nn.Parameter(\n            torch.empty(d_model)\n        )\n        self.delta_bias = nn.Parameter(\n            torch.zeros(d_model)\n        )\n\n        self.total_feature_tokens = (\n            self.categorical_dim\n            + self.numerical_dim\n            + self.binary_dim\n            + 1\n        )\n\n        self.field_embedding = nn.Embedding(\n            self.total_feature_tokens,\n            d_model,\n        )\n        self.type_embedding = nn.Embedding(4, d_model)\n\n        field_type_ids = (\n            [self.TYPE_CAT] * self.categorical_dim\n            + [self.TYPE_NUM] * self.numerical_dim\n            + [self.TYPE_BIN] * self.binary_dim\n            + [self.TYPE_DELTA]\n        )\n\n        self.register_buffer(\n            "field_type_ids",\n            torch.tensor(field_type_ids, dtype=torch.long),\n            persistent=False,\n        )\n\n        self.blocks = nn.ModuleList([\n            SelfAttentionBlock(\n                d_model=d_model,\n                num_heads=num_heads,\n                ff_dim=ff_dim,\n                dropout=dropout,\n            )\n            for _ in range(num_layers)\n        ])\n\n        self.output_norm = nn.LayerNorm(d_model)\n\n        nn.init.normal_(self.num_weight, mean=0.0, std=0.02)\n        nn.init.normal_(\n            self.num_missing_vector,\n            mean=0.0,\n            std=0.02,\n        )\n        nn.init.normal_(self.delta_weight, mean=0.0, std=0.02)\n\n    def build_feature_tokens(\n        self,\n        *,\n        x_cat: torch.Tensor,\n        x_num: torch.Tensor,\n        x_num_missing: torch.Tensor,\n        x_bin: torch.Tensor,\n        delta_t_log1p: torch.Tensor,\n    ) -> torch.Tensor:\n        cat_tokens = torch.stack(\n            [\n                emb(x_cat[:, i])\n                for i, emb in enumerate(self.cat_embeddings)\n            ],\n            dim=1,\n        )\n\n        num_tokens = (\n            x_num.unsqueeze(-1)\n            * self.num_weight.unsqueeze(0)\n            + self.num_bias.unsqueeze(0)\n        )\n\n        num_tokens = (\n            num_tokens\n            + x_num_missing.unsqueeze(-1)\n            * self.num_missing_vector.unsqueeze(0)\n        )\n\n        bin_token = self.binary_embedding(\n            x_bin[:, 0].long().clamp(0, 1)\n        ).unsqueeze(1)\n\n        delta_token = (\n            delta_t_log1p[:, 0].unsqueeze(-1)\n            * self.delta_weight.unsqueeze(0)\n            + self.delta_bias.unsqueeze(0)\n        ).unsqueeze(1)\n\n        tokens = torch.cat(\n            [\n                cat_tokens,\n                num_tokens,\n                bin_token,\n                delta_token,\n            ],\n            dim=1,\n        )\n\n        field_ids = torch.arange(\n            self.total_feature_tokens,\n            device=tokens.device,\n        )\n\n        tokens = (\n            tokens\n            + self.field_embedding(field_ids)[None, :, :]\n            + self.type_embedding(\n                self.field_type_ids\n            )[None, :, :]\n        )\n\n        return tokens\n\n    def forward(\n        self,\n        *,\n        x_cat: torch.Tensor,\n        x_num: torch.Tensor,\n        x_num_missing: torch.Tensor,\n        x_bin: torch.Tensor,\n        delta_t_log1p: torch.Tensor,\n        return_attention: bool = False,\n    ):\n        tokens = self.build_feature_tokens(\n            x_cat=x_cat,\n            x_num=x_num,\n            x_num_missing=x_num_missing,\n            x_bin=x_bin,\n            delta_t_log1p=delta_t_log1p,\n        )\n\n        last_attention = None\n\n        for i, block in enumerate(self.blocks):\n            tokens, weights = block(\n                tokens,\n                return_attention=(\n                    return_attention\n                    and i == len(self.blocks) - 1\n                ),\n            )\n            if weights is not None:\n                last_attention = weights\n\n        tokens = self.output_norm(tokens)\n\n        # Feature-level classification-style pooling.\n        event_embedding = tokens.mean(dim=1)\n\n        return event_embedding, last_attention\n\n\nclass MixedTypeSelfAttentionClassifier(nn.Module):\n    def __init__(\n        self,\n        config: MixedTypeModelConfig,\n    ):\n        super().__init__()\n        self.config = config\n\n        self.event_encoder = MixedTypeEventEncoder(\n            categorical_cardinalities=\n                config.categorical_cardinalities,\n            numerical_dim=\n                config.numerical_dim,\n            binary_dim=\n                config.binary_dim,\n            d_model=\n                config.d_model,\n            num_heads=\n                config.num_heads,\n            num_layers=\n                config.feature_layers,\n            ff_dim=\n                config.ff_dim,\n            dropout=\n                config.dropout,\n        )\n\n        self.position_embedding = nn.Embedding(\n            config.max_events,\n            config.d_model,\n        )\n\n        self.temporal_blocks = nn.ModuleList([\n            SelfAttentionBlock(\n                d_model=config.d_model,\n                num_heads=config.num_heads,\n                ff_dim=config.ff_dim,\n                dropout=config.dropout,\n            )\n            for _ in range(config.temporal_layers)\n        ])\n\n        self.temporal_norm = nn.LayerNorm(config.d_model)\n\n        self.classifier = nn.Sequential(\n            nn.Linear(config.d_model, config.d_model),\n            nn.GELU(),\n            nn.Dropout(config.dropout),\n            nn.Linear(config.d_model, 1),\n        )\n\n    def forward(\n        self,\n        *,\n        x_cat: torch.Tensor,\n        x_num: torch.Tensor,\n        x_num_missing: torch.Tensor,\n        x_bin: torch.Tensor,\n        delta_t_log1p: torch.Tensor,\n        attention_mask: torch.Tensor,\n        return_attention: bool = False,\n    ):\n        b, t, _ = x_cat.shape\n\n        if t > self.config.max_events:\n            raise ValueError(\n                f"T={t} exceeds max_events={self.config.max_events}"\n            )\n\n        valid = attention_mask.bool()\n        flat_valid = valid.reshape(-1)\n\n        flat_cat = x_cat.reshape(\n            b * t,\n            x_cat.shape[-1],\n        )[flat_valid]\n\n        flat_num = x_num.reshape(\n            b * t,\n            x_num.shape[-1],\n        )[flat_valid]\n\n        flat_missing = x_num_missing.reshape(\n            b * t,\n            x_num_missing.shape[-1],\n        )[flat_valid]\n\n        flat_bin = x_bin.reshape(\n            b * t,\n            x_bin.shape[-1],\n        )[flat_valid]\n\n        flat_delta = delta_t_log1p.reshape(\n            b * t,\n            1,\n        )[flat_valid]\n\n        event_valid, feature_attention = self.event_encoder(\n            x_cat=flat_cat,\n            x_num=flat_num,\n            x_num_missing=flat_missing,\n            x_bin=flat_bin,\n            delta_t_log1p=flat_delta,\n            return_attention=return_attention,\n        )\n\n        event_embeddings = torch.zeros(\n            b * t,\n            self.config.d_model,\n            device=x_cat.device,\n            dtype=event_valid.dtype,\n        )\n\n        event_embeddings[flat_valid] = event_valid\n\n        event_embeddings = event_embeddings.view(\n            b,\n            t,\n            self.config.d_model,\n        )\n\n        positions = torch.arange(\n            t,\n            device=x_cat.device,\n        )\n\n        temporal = (\n            event_embeddings\n            + self.position_embedding(positions)[None, :, :]\n        )\n\n        key_padding_mask = ~valid\n        last_temporal_attention = None\n\n        for i, block in enumerate(self.temporal_blocks):\n            temporal, weights = block(\n                temporal,\n                key_padding_mask=key_padding_mask,\n                return_attention=(\n                    return_attention\n                    and i == len(self.temporal_blocks) - 1\n                ),\n            )\n            if weights is not None:\n                last_temporal_attention = weights\n\n        temporal = self.temporal_norm(temporal)\n\n        mask_float = valid.unsqueeze(-1).to(temporal.dtype)\n\n        pooled = (\n            (temporal * mask_float).sum(dim=1)\n            / mask_float.sum(dim=1).clamp_min(1.0)\n        )\n\n        logits = self.classifier(pooled).squeeze(-1)\n\n        if return_attention:\n            return {\n                "logits": logits,\n                "feature_attention": feature_attention,\n                "temporal_attention": last_temporal_attention,\n                "valid_event_mask": valid,\n            }\n\n        return logits\n', encoding='utf-8')
(WORK_CODE/'s14_ft_train.py').write_text('#!/usr/bin/env python3\n"""S14b - Fine-tune mô hình B (BiLSTM hoặc Mixed-Type) trên hai miền Windows (skrghosh, FiveDirections) với từ điển mở rộng và replay OpTC (STEP 30). Chạy trên Kaggle; chạy thử CPU được.\n\nMỗi fold k:\n  1. dựng mô hình với bảng embedding mở rộng (hàng mới = bản sao hàng <UNK>); các tham số khác copy từ mô hình gốc; kiến trúc không đổi.\n  2. KIỂM TRA BẮT BUỘC: trước khi train, logit trên mẫu test của fold phải trùng dự đoán zero-shot tham chiếu (STEP 26, 29) trong dung sai; không qua thì dừng.\n  3. giai đoạn A: chỉ học hàng embedding mới + đầu phân loại; giai đoạn B: mở khóa toàn bộ, lr nhỏ. Mỗi epoch = toàn bộ mẫu TRAIN của hai miền (fold != k) + một lần rút mới từ tập replay OpTC.\n  4. đánh giá MỘT lần: mẫu test của fold (hai miền) và OpTC TEST (tuỳ chọn); ngưỡng cố định: ngưỡng VAL OpTC của mô hình gốc và 0.5. Không chọn ngưỡng, epoch hay siêu tham số theo dữ liệu mới.\nĐầu vào mô hình: 4 category (app_family, proto, resp_port_class, image_dir_class) + delta_t_log1p; numeric rỗng; x_bin = 0 (giống bản B)."""\nimport argparse\nimport importlib.util\nimport json\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\nCAT_IDX = [0, 1, 4, 5]  # app_family, proto, resp_port_class, image_dir_class trong x_cat 7 cột của shards\nCAT_NAMES = ["app_family", "proto", "resp_port_class", "image_dir_class"]\nEMB_PREFIX = {"bilstm": "embeddings.", "mixed": "event_encoder.cat_embeddings."}\n\n\ndef load_module(path, name):\n    sys.path.insert(0, str(Path(path).parent))  # 05b_train_bilstm.py import network_dl_dataset_v1_1 cùng thư mục\n    spec = importlib.util.spec_from_file_location(name, path)\n    mod = importlib.util.module_from_spec(spec)\n    sys.modules[name] = mod\n    spec.loader.exec_module(mod)\n    return mod\n\n\n# ---------------------------------------------------------------- dữ liệu\ndef read_shard(path):\n    with np.load(path, allow_pickle=False) as z:\n        return {k: z[k] for k in ("x_cat", "delta_t_log1p", "sample_offsets")}\n\n\ndef sample_from(arrs, i, label, sid):\n    off = arrs["sample_offsets"]\n    a, b = int(off[i]), int(off[i + 1])\n    return {"x_cat": arrs["x_cat"][a:b][:, CAT_IDX].astype(np.int64), "delta": arrs["delta_t_log1p"][a:b].astype(np.float32), "label": int(label), "sample_id": sid}\n\n\ndef materialize(rows):\n    """rows: DataFrame có cột root, shard_file, shard_sample_index, label, sample_id -> list sample (giữ thứ tự rows)."""\n    out = [None] * len(rows)\n    rows = rows.reset_index(drop=True)\n    for (root, shard), g in rows.groupby(["root", "shard_file"], sort=False):\n        arrs = read_shard(Path(root) / shard)\n        for pos, r in zip(g.index, g.itertuples()):\n            out[pos] = sample_from(arrs, int(r.shard_sample_index), r.label, r.sample_id)\n    return out\n\n\ndef collate(batch):\n    lengths = torch.tensor([len(s["delta"]) for s in batch], dtype=torch.long)\n    T, B = int(lengths.max()), len(batch)\n    x_cat = torch.zeros(B, T, len(CAT_IDX), dtype=torch.long)\n    delta = torch.zeros(B, T, 1)\n    mask = torch.zeros(B, T, dtype=torch.bool)\n    for i, s in enumerate(batch):\n        t = len(s["delta"])\n        x_cat[i, :t] = torch.from_numpy(s["x_cat"])\n        delta[i, :t, 0] = torch.from_numpy(s["delta"])\n        mask[i, :t] = True\n    return {"x_cat": x_cat, "x_num": torch.zeros(B, T, 0), "x_num_missing": torch.zeros(B, T, 0), "x_bin": torch.zeros(B, T, 1), "delta_t_log1p": delta, "mask": mask,\n            "lengths": lengths, "labels": torch.tensor([s["label"] for s in batch], dtype=torch.float32), "sample_ids": [s["sample_id"] for s in batch]}\n\n\ndef forward(model, kind, b, device):\n    kw = {k: b[k].to(device) for k in ("x_cat", "x_num", "x_num_missing", "x_bin", "delta_t_log1p")}\n    if kind == "bilstm":\n        return model(lengths=b["lengths"], **kw)\n    return model(attention_mask=b["mask"].to(device), **kw)\n\n\n@torch.no_grad()\ndef predict(model, kind, samples, device, bs):\n    model.eval()\n    logit, labels, ids = [], [], []\n    for i in range(0, len(samples), bs):\n        b = collate(samples[i:i + bs])\n        logit.append(forward(model, kind, b, device).float().cpu().numpy())\n        labels.append(b["labels"].numpy())\n        ids += b["sample_ids"]\n    logit = np.concatenate(logit) if logit else np.zeros(0)\n    prob = torch.sigmoid(torch.from_numpy(logit).half()).float().numpy()  # giống mô phỏng fp16 ở STEP 26\n    return pd.DataFrame({"sample_id": ids, "label": (np.concatenate(labels) if labels else np.zeros(0)).astype(int), "logit": logit, "probability": prob})\n\n\ndef metrics_at(y, p, thr):\n    pred = (p >= thr).astype(int)\n    tp = int(((y == 1) & (pred == 1)).sum()); fp = int(((y == 0) & (pred == 1)).sum()); fn = int(((y == 1) & (pred == 0)).sum()); tn = int(((y == 0) & (pred == 0)).sum())\n    pr = tp / (tp + fp) if tp + fp else 0.0\n    rc = tp / (tp + fn) if tp + fn else 0.0\n    f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0\n    return {"threshold": float(thr), "tp": tp, "fp": fp, "fn": fn, "tn": tn, "precision": round(pr, 4), "recall": round(rc, 4), "f1": round(f1, 4)}\n\n\ndef summarize(df, thresholds):\n    y, p, lg = df.label.to_numpy(), df.probability.to_numpy(), df.logit.to_numpy()\n    out = {"n": int(len(df)), "positives": int(y.sum()), "at": {name: metrics_at(y, p, t) for name, t in thresholds.items()}}\n    if 0 < y.sum() < len(y):\n        out["pr_auc"] = round(float(average_precision_score(y, lg)), 4)\n        out["roc_auc"] = round(float(roc_auc_score(y, lg)), 4)\n    return out\n\n\n# ---------------------------------------------------------------- mô hình\ndef build_model(kind, cfg, cards, code_dir):\n    if kind == "bilstm":\n        m = load_module(Path(code_dir) / "05b_train_bilstm.py", "train_bilstm")\n        base_dims = [m.embedding_dim(c) for c in cfg["categorical_cardinalities"]]  # giữ nguyên chiều embedding của mô hình gốc dù số hàng tăng (hàm gốc suy chiều từ số hàng)\n        it = iter(base_dims)\n        m.embedding_dim = lambda c: next(it)\n        return m.BiLSTMBaseline(categorical_cardinalities=cards, numerical_dim=0, event_dim=cfg["event_dim"], hidden_dim=cfg["hidden_dim"], lstm_layers=cfg["lstm_layers"], dropout=cfg["dropout"])\n    m = load_module(Path(code_dir) / "mixed_type_attention_model.py", "mixed_model")\n    return m.MixedTypeSelfAttentionClassifier(m.MixedTypeModelConfig(categorical_cardinalities=cards, numerical_dim=0, binary_dim=1, max_events=cfg["max_events"], d_model=cfg["d_model"],\n                                                                      num_heads=cfg["num_heads"], feature_layers=cfg["feature_layers"], temporal_layers=cfg["temporal_layers"], ff_dim=cfg["ff_dim"],\n                                                                      dropout=cfg["dropout"]))\n\n\ndef transfer_weights(model, base_sd, old_cards, kind):\n    """Copy tham số gốc; hàng embedding mới = bản sao hàng <UNK> (id 0). Trả về {tên tham số embedding: số hàng cũ}."""\n    sd = model.state_dict()\n    new_sd, old_rows = {}, {}\n    for key, tensor in sd.items():\n        if key not in base_sd:\n            raise KeyError(f"tham số {key} không có trong mô hình gốc")\n        base = base_sd[key]\n        if key.startswith(EMB_PREFIX[kind]) and key.endswith(".weight"):\n            idx = int(key[len(EMB_PREFIX[kind]):].split(".")[0])\n            assert base.shape[1] == tensor.shape[1], f"chiều embedding đổi ở {key}: {tuple(base.shape)} -> {tuple(tensor.shape)}"\n            assert base.shape[0] == old_cards[idx] and tensor.shape[0] >= base.shape[0]\n            w = tensor.clone()\n            w[:base.shape[0]] = base\n            w[base.shape[0]:] = base[0:1]\n            new_sd[key] = w\n            old_rows[key] = base.shape[0]\n        else:\n            assert base.shape == tensor.shape, (key, base.shape, tensor.shape)\n            new_sd[key] = base\n    model.load_state_dict(new_sd)\n    return old_rows\n\n\ndef set_stage(model, kind, stage, old_rows):\n    """A: chỉ hàng embedding mới và đầu phân loại. B: tất cả. Trả về danh sách hook cần gỡ."""\n    handles = []\n    for name, p in model.named_parameters():\n        if stage == "B":\n            p.requires_grad = True\n        elif name in old_rows:\n            p.requires_grad = True\n            mask = torch.zeros(p.shape[0], 1, device=p.device)\n            mask[old_rows[name]:] = 1.0\n            handles.append(p.register_hook(lambda g, m=mask: g * m))\n        else:\n            p.requires_grad = name.startswith("classifier.")\n    return handles\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\n# ---------------------------------------------------------------- chạy một fold\ndef run_fold(args, k, ctx):\n    dev = ctx["device"]\n    kind = args.model\n    ft = Path(args.ft_root)\n    fold_dir = ft / f"fold{k}"\n    spec = json.load(open(fold_dir / "spec_ext.json", encoding="utf-8"))\n    cards = [max(e["id"] for e in spec["categorical"]["fields"][f]["vocab"]) + 1 for f in CAT_NAMES]\n    cfg, old_cards = ctx["cfg"], ctx["cfg"]["categorical_cardinalities"]\n    print(f"[fold {k}] cardinalities cũ {old_cards} -> mở rộng {cards}", flush=True)\n    seed_everything(args.seed + k)\n    model = build_model(kind, cfg, cards, args.code_dir)\n    old_rows = transfer_weights(model, ctx["base_sd"], old_cards, kind)\n    model.to(dev)\n    # dữ liệu hai miền\n    folds = pd.read_csv(ft / "folds.csv")\n    rows = []\n    for ds in ctx["domains"]:\n        sm = pd.read_csv(fold_dir / ds / "data/samples.csv")[["sample_id", "shard_file", "shard_sample_index"]]\n        m = sm.merge(folds[folds.dataset == ds][["sample_id", "fold", "label"]], on="sample_id")\n        m["root"] = str(fold_dir / ds)\n        m["dataset"] = ds\n        rows.append(m)\n    rows = pd.concat(rows, ignore_index=True)\n    samples = materialize(rows)\n    tr_idx = [i for i in range(len(rows)) if rows.fold.iat[i] != k]\n    te_idx = [i for i in range(len(rows)) if rows.fold.iat[i] == k]\n    train_new = [samples[i] for i in tr_idx]\n    test_samples = [samples[i] for i in te_idx]\n    print(f"[fold {k}] train {len(train_new)} mẫu (dương {sum(s[\'label\'] for s in train_new)}), test {len(test_samples)} (dương {sum(s[\'label\'] for s in test_samples)})", flush=True)\n    thr = {"optc_val_threshold": ctx["thr"], "0.5": 0.5}\n    # kiểm tra khớp zero-shot\n    pre = predict(model, kind, test_samples, dev, args.eval_batch)\n    pre["dataset"] = rows.dataset.iloc[te_idx].to_numpy()\n    worst = 0.0\n    for ds in ctx["domains"]:\n        ref = pd.read_csv(ft / "ref_zeroshot" / f"predictions_{ds}_{kind}.csv")[["sample_id", "logit"]].rename(columns={"logit": "ref_logit"})\n        m = pre[pre.dataset == ds].merge(ref, on="sample_id")\n        assert len(m) == (pre.dataset == ds).sum(), f"thiếu mẫu tham chiếu {ds}"\n        d = float((m.logit - m.ref_logit).abs().max())\n        worst = max(worst, d)\n        print(f"[fold {k}] khớp zero-shot {ds}: chênh logit lớn nhất {d:.5f} trên {len(m)} mẫu", flush=True)\n    if worst > args.equiv_tol:\n        raise RuntimeError(f"mô hình mở rộng chưa train KHÔNG khớp zero-shot (chênh {worst:.5f} > {args.equiv_tol}); dừng")\n    result = {"fold": k, "cardinalities_old": old_cards, "cardinalities_ext": cards, "equivalence_max_logit_diff": worst,\n              "zero_shot_on_test": summarize(pre, thr), "added_tokens": spec["extension"]["added"], "history": []}\n    # replay\n    rep = ctx["replay"]\n    rep_pos = [s for s in rep if s["label"] == 1]\n    rep_neg = [s for s in rep if s["label"] == 0]\n    n_pos = sum(s["label"] for s in train_new) + args.replay_pos\n    n_neg = len(train_new) - sum(s["label"] for s in train_new) + (args.replay_per_epoch - args.replay_pos)\n    pos_weight = torch.tensor(n_neg / max(n_pos, 1), dtype=torch.float32, device=dev)\n    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n    amp = dev.type == "cuda" and not args.no_amp\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n    rng = np.random.default_rng(args.seed * 1000 + k)\n    ep_global = 0\n    for stage, epochs, lr, wd in (("A", args.epochs_a, args.lr_a, 0.0), ("B", args.epochs_b, args.lr_b_bilstm if kind == "bilstm" else args.lr_b_mixed, args.weight_decay)):\n        handles = set_stage(model, kind, stage, old_rows)\n        params = [p for p in model.parameters() if p.requires_grad]\n        opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)\n        print(f"[fold {k}] giai đoạn {stage}: {epochs} epoch, lr {lr}, tham số học {sum(p.numel() for p in params):,}", flush=True)\n        for ep in range(epochs):\n            ep_global += 1\n            draw = [rep_pos[i] for i in rng.permutation(len(rep_pos))[:args.replay_pos]] + [rep_neg[i] for i in rng.permutation(len(rep_neg))[:args.replay_per_epoch - args.replay_pos]]\n            epoch_samples = train_new + draw\n            order = rng.permutation(len(epoch_samples))\n            model.train()\n            tot, nb, t0 = 0.0, 0, time.time()\n            for i in range(0, len(order), args.batch_size):\n                b = collate([epoch_samples[j] for j in order[i:i + args.batch_size]])\n                opt.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):\n                    loss = crit(forward(model, kind, b, dev).float(), b["labels"].to(dev))\n                scaler.scale(loss).backward()\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(params, args.grad_clip)\n                scaler.step(opt)\n                scaler.update()\n                tot += float(loss.item())\n                nb += 1\n            result["history"].append({"stage": stage, "epoch": ep_global, "train_loss": round(tot / max(nb, 1), 5), "seconds": round(time.time() - t0, 1)})\n            print(f"[fold {k}] {stage} epoch {ep_global}: loss {tot / max(nb, 1):.5f} ({time.time() - t0:.0f}s)", flush=True)\n        for h in handles:\n            h.remove()\n    # đánh giá một lần\n    post = predict(model, kind, test_samples, dev, args.eval_batch)\n    post["dataset"] = rows.dataset.iloc[te_idx].to_numpy()\n    post["fold"] = k\n    result["after_on_test"] = summarize(post, thr)\n    result["after_on_test_by_dataset"] = {ds: summarize(post[post.dataset == ds], thr) for ds in ctx["domains"]}\n    result["before_on_test_by_dataset"] = {ds: summarize(pre[pre.dataset == ds], thr) for ds in ctx["domains"]}\n    out = Path(args.out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    post.to_csv(out / f"predictions_{kind}_fold{k}_newdomains.csv", index=False)\n    torch.save({"model_state_dict": model.state_dict(), "cardinalities": cards, "fold": k}, out / f"model_{kind}_fold{k}.pt")\n    if args.optc_eval != "none":\n        result["optc_test"] = eval_optc(model, kind, args, ctx, thr, k)\n    json.dump(result, open(out / f"result_{kind}_fold{k}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)\n    print(f"[fold {k}] xong; test sau fine-tune:", json.dumps(result["after_on_test"]["at"], ensure_ascii=False), flush=True)\n    return result\n\n\ndef eval_optc(model, kind, args, ctx, thr, k):\n    """OpTC TEST (đọc streaming theo shard); với mô hình mở rộng id cũ < 61 nên dùng nguyên shards OpTC."""\n    t0 = time.time()\n    man = ctx["optc_manifest"]\n    te = man[man.split == "test"]\n    if args.max_optc_test:\n        te = pd.concat([te[te.label == 1], te[te.label == 0].head(args.max_optc_test)])\n    root = Path(args.optc_root)\n    frames = []\n    for shard, g in te.groupby("shard_file", sort=False):\n        arrs = read_shard(root / shard)\n        samples = [sample_from(arrs, int(r.shard_sample_index), r.label, r.sample_id) for r in g.itertuples()]\n        frames.append(predict(model, kind, samples, ctx["device"], args.eval_batch))\n    df = pd.concat(frames, ignore_index=True)\n    df.to_csv(Path(args.out_dir) / f"predictions_{kind}_fold{k}_optc_test.csv", index=False)\n    res = summarize(df, thr)\n    res["seconds"] = round(time.time() - t0)\n    print(f"[fold {k}] OpTC TEST: {json.dumps(res[\'at\'], ensure_ascii=False)} ({res[\'seconds\']}s)", flush=True)\n    return res\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model", choices=["bilstm", "mixed"], required=True)\n    ap.add_argument("--ft-root", required=True, help="gói v3_ft_kaggle_upload (có ft_manifest.json)")\n    ap.add_argument("--optc-root", required=True, help="gốc dataset OpTC v2 (split/, shards/)")\n    ap.add_argument("--code-dir", required=True, help="thư mục chứa 05b_train_bilstm.py và mixed_type_attention_model.py")\n    ap.add_argument("--out-dir", required=True)\n    ap.add_argument("--folds", default="0,1,2")\n    ap.add_argument("--seed", type=int, default=42)\n    ap.add_argument("--epochs-a", type=int, default=3)\n    ap.add_argument("--epochs-b", type=int, default=3)\n    ap.add_argument("--lr-a", type=float, default=1e-3)\n    ap.add_argument("--lr-b-bilstm", type=float, default=1e-4)\n    ap.add_argument("--lr-b-mixed", type=float, default=3e-5)\n    ap.add_argument("--weight-decay", type=float, default=1e-4)\n    ap.add_argument("--grad-clip", type=float, default=1.0)\n    ap.add_argument("--replay-per-epoch", type=int, default=600)\n    ap.add_argument("--replay-pos", type=int, default=150)\n    ap.add_argument("--batch-size", type=int, default=0, help="0 = mặc định theo mô hình (BiLSTM 32, Mixed 16)")\n    ap.add_argument("--eval-batch", type=int, default=0)\n    ap.add_argument("--equiv-tol", type=float, default=0.02)\n    ap.add_argument("--optc-eval", choices=["test", "none"], default="test")\n    ap.add_argument("--max-optc-test", type=int, default=0, help="chỉ để chạy thử: giới hạn số mẫu âm OpTC TEST")\n    ap.add_argument("--no-amp", action="store_true")\n    ap.add_argument("--cpu", action="store_true")\n    args = ap.parse_args()\n    args.batch_size = args.batch_size or (32 if args.model == "bilstm" else 16)\n    args.eval_batch = args.eval_batch or (128 if args.model == "bilstm" else 32)\n    dev = torch.device("cpu" if args.cpu or not torch.cuda.is_available() else "cuda")\n    ft = Path(args.ft_root)\n    man = json.load(open(ft / "ft_manifest.json", encoding="utf-8"))\n    base = ft / "base_models" / args.model\n    cfg = json.load(open(base / "config.json", encoding="utf-8"))\n    thr = float(json.load(open(base / "threshold.json", encoding="utf-8"))["threshold"])\n    ck = torch.load(base / "best_model.pt", map_location="cpu", weights_only=False)\n    print(f"[init] mô hình {args.model}, thiết bị {dev}, ngưỡng VAL OpTC {thr}, thư mục ra {args.out_dir}", flush=True)\n    optc_manifest = pd.read_csv(Path(args.optc_root) / "split/network_sample_manifest.csv")\n    # tập replay (đọc một lần)\n    rr = pd.read_csv(ft / "replay_R.csv")\n    rr["root"] = str(Path(args.optc_root))\n    t0 = time.time()\n    replay = materialize(rr)\n    print(f"[init] replay R: {len(replay)} mẫu (dương {sum(s[\'label\'] for s in replay)}) nạp trong {time.time() - t0:.0f}s", flush=True)\n    ctx = {"device": dev, "cfg": cfg, "base_sd": ck["model_state_dict"], "thr": thr, "domains": man["domains"], "replay": replay, "optc_manifest": optc_manifest}\n    results = []\n    for k in [int(x) for x in args.folds.split(",")]:\n        results.append(run_fold(args, k, ctx))\n    json.dump({"model": args.model, "args": vars(args), "folds": [r["fold"] for r in results]}, open(Path(args.out_dir) / f"run_{args.model}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)\n    print("[xong]", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(WORK_CODE/'s15_train_diag.py').write_text('#!/usr/bin/env python3\n"""S15 - Chẩn đoán CHỈ TRÊN TRAIN (STEP 34): cho mô hình B học lâu hơn / lr cao hơn trên TRAIN của một fold, xem có fit được các mẫu dương TRAIN không.\n\nPhân biệt (a) "đầu vào không đủ tách" với (b) "huấn luyện quá ngắn / lr quá nhỏ" của STEP 33. Không đọc mẫu test của fold, không đánh giá OpTC TEST, không chọn gì theo dữ liệu đó.\nMô hình khởi tạo giống s14_ft_train (mô hình gốc + từ điển mở rộng của fold, hàng mới = bản sao <UNK>), mở khóa toàn bộ tham số ngay từ đầu.\nInput : 03_datasets/v3_ft_kaggle_upload (folds, fold<k>, base_models, replay_R.csv), 03_datasets/v2_network_dl_kaggle_upload (shards OpTC cho replay, nếu --replay), mã mô hình trích từ notebook subsetB\nOutput: --out-dir/diag_<model>_fold<k>_<tag>.json (đường cong theo epoch) và .csv (xác suất cuối từng mẫu TRAIN)\nCách dùng: python s15_train_diag.py --model bilstm --fold 0 --epochs 60 --lr 1e-3 --replay 0 --tag e60"""\nimport argparse\nimport json\nimport sys\nimport tempfile\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\nsys.stdout.reconfigure(encoding="utf-8")\nsys.path.insert(0, str(Path(__file__).parent))\nimport s14_ft_train as T  # noqa: E402\n\nROOT = Path("D:/UIT/KLTNv2")\nFT = ROOT / "03_datasets/v3_ft_kaggle_upload"\n\n\ndef fit_stats(pr, thr):\n    y, p, lg = pr.label.to_numpy(), pr.probability.to_numpy(), pr.logit.to_numpy()\n    out = {}\n    for name, t in (("0.5", 0.5), ("optc_val", thr)):\n        pred = p >= t\n        out[name] = {"tp": int((pred & (y == 1)).sum()), "fp": int((pred & (y == 0)).sum())}\n    out["roc_auc"] = round(float(roc_auc_score(y, lg)), 4)\n    out["pr_auc"] = round(float(average_precision_score(y, lg)), 4)\n    out["median_prob_pos"] = round(float(np.median(p[y == 1])), 4)\n    out["median_prob_neg"] = round(float(np.median(p[y == 0])), 5)\n    return out\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model", choices=["bilstm", "mixed"], required=True)\n    ap.add_argument("--fold", type=int, default=0)\n    ap.add_argument("--epochs", type=int, default=60)\n    ap.add_argument("--lr", type=float, required=True)\n    ap.add_argument("--weight-decay", type=float, default=1e-4)\n    ap.add_argument("--replay", type=int, default=0, help="số mẫu replay OpTC mỗi epoch (0 = không dùng); nếu >0, một phần tư là dương")\n    ap.add_argument("--eval-every", type=int, default=5)\n    ap.add_argument("--batch-size", type=int, default=0)\n    ap.add_argument("--seed", type=int, default=42)\n    ap.add_argument("--max-epoch-seconds", type=float, default=0, help="dừng nếu một epoch quá lâu (chỉ để chạy thử)")\n    ap.add_argument("--tag", default="diag")\n    ap.add_argument("--out-dir", default=str(ROOT / "02_artifacts/34_train_diag_v1/runs"))\n    ap.add_argument("--threads", type=int, default=0)\n    ap.add_argument("--ft-root", default=str(FT))\n    ap.add_argument("--optc-root", default=str(ROOT / "03_datasets/v2_network_dl_kaggle_upload"), help="gốc dataset OpTC v2 (chỉ cần khi --replay > 0)")\n    ap.add_argument("--code-dir", default="", help="thư mục chứa 05b_train_bilstm.py / mixed_type_attention_model.py (Kaggle); để trống = trích từ notebook subsetB (local)")\n    ap.add_argument("--cpu", action="store_true")\n    args = ap.parse_args()\n    if args.threads:\n        torch.set_num_threads(args.threads)\n    kind, k = args.model, args.fold\n    out = Path(args.out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    bs = args.batch_size or (32 if kind == "bilstm" else 16)\n    dev = torch.device("cpu" if args.cpu or not torch.cuda.is_available() else "cuda")\n    FTR = Path(args.ft_root)\n    if args.code_dir:\n        code = Path(args.code_dir)\n    else:  # mã mô hình trích từ notebook B, như s14_collect\n        from s8_notebooks_subset import extract_files\n        code = Path(tempfile.mkdtemp(prefix="s15_code_"))\n        nbname = "06e_BiLSTM_Kaggle_v2_subsetB.ipynb" if kind == "bilstm" else "07e_Mixed_Type_Kaggle_v2_subsetB.ipynb"\n        d, _ = extract_files(json.load(open(ROOT / "04_notebooks" / nbname, encoding="utf-8")), "c")\n        for p in d.iterdir():\n            (code / p.name).write_text(p.read_text(encoding="utf-8"), encoding="utf-8")\n    base = FTR / "base_models" / kind\n    cfg = json.load(open(base / "config.json", encoding="utf-8"))\n    thr = float(json.load(open(base / "threshold.json", encoding="utf-8"))["threshold"])\n    ck = torch.load(base / "best_model.pt", map_location="cpu", weights_only=False)\n    spec = json.load(open(FTR / f"fold{k}/spec_ext.json", encoding="utf-8"))\n    cards = [max(e["id"] for e in spec["categorical"]["fields"][f]["vocab"]) + 1 for f in T.CAT_NAMES]\n    T.seed_everything(args.seed + k)\n    model = T.build_model(kind, cfg, cards, code)\n    T.transfer_weights(model, ck["model_state_dict"], cfg["categorical_cardinalities"], kind)\n    # TRAIN của fold (fold != k); KHÔNG nạp mẫu test của fold\n    folds = pd.read_csv(FTR / "folds.csv")\n    rows = []\n    for ds in ("skrghosh", "fivedirections"):\n        sm = pd.read_csv(FTR / f"fold{k}/{ds}/data/samples.csv")[["sample_id", "shard_file", "shard_sample_index"]]\n        m = sm.merge(folds[folds.dataset == ds][["sample_id", "fold", "label"]], on="sample_id")\n        m["root"] = str(FTR / f"fold{k}/{ds}")\n        m["dataset"] = ds\n        rows.append(m)\n    rows = pd.concat(rows, ignore_index=True)\n    rows = rows[rows.fold != k].reset_index(drop=True)\n    train = T.materialize(rows)\n    n_pos = sum(s["label"] for s in train)\n    print(f"[diag] {kind} fold {k}: TRAIN {len(train)} mẫu (dương {n_pos}), lr {args.lr}, epoch {args.epochs}, replay {args.replay}, batch {bs}, threads {torch.get_num_threads()}", flush=True)\n    rep_pos = rep_neg = []\n    if args.replay:\n        rr = pd.read_csv(FTR / "replay_R.csv")\n        rr["root"] = args.optc_root\n        rep = T.materialize(rr)\n        rep_pos = [s for s in rep if s["label"] == 1]\n        rep_neg = [s for s in rep if s["label"] == 0]\n    rp = args.replay // 4\n    npos = n_pos + (rp if args.replay else 0)\n    nneg = len(train) - n_pos + ((args.replay - rp) if args.replay else 0)\n    model.to(dev)\n    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(nneg / max(npos, 1), dtype=torch.float32, device=dev))\n    for p in model.parameters():\n        p.requires_grad = True\n    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    rng = np.random.default_rng(args.seed * 1000 + k)\n    curve = []\n    t_all = time.time()\n    for ep in range(1, args.epochs + 1):\n        draw = []\n        if args.replay:\n            draw = [rep_pos[i] for i in rng.permutation(len(rep_pos))[:rp]] + [rep_neg[i] for i in rng.permutation(len(rep_neg))[:args.replay - rp]]\n        es = train + draw\n        order = rng.permutation(len(es))\n        model.train()\n        tot, nb, t0 = 0.0, 0, time.time()\n        for i in range(0, len(order), bs):\n            b = T.collate([es[j] for j in order[i:i + bs]])\n            opt.zero_grad(set_to_none=True)\n            loss = crit(T.forward(model, kind, b, dev).float(), b["labels"].to(dev))\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            opt.step()\n            tot += float(loss.item())\n            nb += 1\n        rec = {"epoch": ep, "train_loss": round(tot / nb, 5), "seconds": round(time.time() - t0, 1)}\n        if ep % args.eval_every == 0 or ep == 1 or ep == args.epochs:\n            pr = T.predict(model, kind, train, dev, 64)\n            rec["fit"] = fit_stats(pr, thr)\n            print(f"[diag] ep {ep}: loss {rec[\'train_loss\']:.4f} ({rec[\'seconds\']}s) TP@0.5 {rec[\'fit\'][\'0.5\'][\'tp\']}/{n_pos} FP@0.5 {rec[\'fit\'][\'0.5\'][\'fp\']} ROC {rec[\'fit\'][\'roc_auc\']} PR {rec[\'fit\'][\'pr_auc\']}", flush=True)\n        else:\n            print(f"[diag] ep {ep}: loss {rec[\'train_loss\']:.4f} ({rec[\'seconds\']}s)", flush=True)\n        curve.append(rec)\n        if args.max_epoch_seconds and rec["seconds"] > args.max_epoch_seconds:\n            print("[diag] dừng: epoch quá lâu", flush=True)\n            break\n    pr = T.predict(model, kind, train, dev, 64)\n    pr["dataset"] = rows.dataset.to_numpy()\n    pr.to_csv(out / f"diag_{kind}_fold{k}_{args.tag}.csv", index=False)\n    res = {"model": kind, "fold": k, "args": vars(args), "train_samples": len(train), "train_positives": n_pos, "optc_threshold": thr, "total_seconds": round(time.time() - t_all), "curve": curve, "final_fit": fit_stats(pr, thr)}\n    json.dump(res, open(out / f"diag_{kind}_fold{k}_{args.tag}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)\n    print("[xong]", json.dumps(res["final_fit"], ensure_ascii=False), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Wrote:', sorted(p.name for p in WORK_CODE.iterdir()))
sys.path.insert(0, str(WORK_CODE))


## Chạy các cấu hình

In [ ]:
OUTPUT_DIR = Path(f'/kaggle/working/{RUN_NAME}')
for tag, lr, replay in CONFIGS:
    cmd = (f"python {WORK_CODE/'s15_train_diag.py'} --model {MODEL} --fold {FOLD} --epochs {EPOCHS} --lr {lr} --replay {replay} --eval-every {EVAL_EVERY} "
           f"--tag {tag} --ft-root '{FT_ROOT}' --optc-root '{DATASET_ROOT}' --code-dir '{WORK_CODE}' --out-dir '{OUTPUT_DIR}'")
    print('\n=====', tag, '=====')
    print(cmd)
    import subprocess
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f'exit code {ret}')


## Tóm tắt

In [ ]:
for p in sorted(OUTPUT_DIR.glob('diag_*.json')):
    r = json.loads(p.read_text())
    print('===', p.name, '| TRAIN', r['train_samples'], 'dương', r['train_positives'], '| giây', r['total_seconds'])
    for c in r['curve']:
        if 'fit' in c:
            f = c['fit']
            print(f"  ep {c['epoch']:>3} loss {c['train_loss']:.4f} TP@0.5 {f['0.5']['tp']}/{r['train_positives']} FP@0.5 {f['0.5']['fp']} ROC {f['roc_auc']} PR {f['pr_auc']}")


In [ ]:
archive = shutil.make_archive(f'/kaggle/working/{RUN_NAME}', 'zip', OUTPUT_DIR)
print('Created:', archive)
